# Week 6 · Thursday: TensorFlow Introduction

Yesterday's network ran in numpy. Today the same kind of arithmetic runs in TensorFlow:

1. **TensorFlow in Colab:** what it is, and which version we are running
2. **Tensors:** rank, shape, dtype and arithmetic
3. **Implementing basic graphs:** one function run eagerly, then as a graph with `tf.function`
4. **Reducing tensors of arbitrary shape:** sums, means and maxes over any axis, on 60,000 handwritten digits
5. **Graphs, sessions, TensorBoard:** how TensorFlow 1 worked, and how to look at a graph today

Run the next cell first. It imports TensorFlow, prints its version and holds the plotting helpers we use this week.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import tensorflow as tf

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)   # seeds Python, NumPy and TensorFlow in one call


def plot_points(X, labels=None, title="", xlabel="Feature 1", ylabel="Feature 2",
                names=None, centers=None, ax=None):
    """Scatter 2D points, coloured by label if given. Label -1 is drawn as noise (DBSCAN)."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], s=45, edgecolor="black", alpha=0.85)
    else:
        for value in np.unique(labels):
            mask = labels == value
            if value == -1:
                ax.scatter(X[mask, 0], X[mask, 1], s=70, marker="x", color="black", label="Noise (-1)")
            else:
                label = names[value] if names is not None else f"{value}"
                ax.scatter(X[mask, 0], X[mask, 1], s=45, edgecolor="black", alpha=0.85, label=label)
        ax.legend()
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], s=300, marker="X", color="red",
                   edgecolor="black", linewidth=2, label="Centroids")
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if show:
        plt.show()


def plot_decision_boundary(model, X, y, title="", xlabel="Feature 1", ylabel="Feature 2",
                           names=None, ax=None):
    """Colour the background by what the model predicts at every point of a fine grid."""
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300))
    grid_predictions = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    classes = np.unique(y)
    colours = ListedColormap(plt.cm.tab10.colors[:len(classes)])   # same colours as the points
    ax.contourf(xx, yy, grid_predictions, levels=np.arange(len(classes) + 1) - 0.5, alpha=0.25, cmap=colours)
    plot_points(X, y, title, xlabel, ylabel, names=names, ax=ax)
    if show:
        plt.show()


def plot_line_chart(xs, ys, title, xlabel, ylabel):
    """A simple line chart with markers, for the elbow and silhouette plots."""
    plt.figure(figsize=(8, 5))
    plt.plot(xs, ys, marker="o", linewidth=2)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(xs)
    plt.grid(alpha=0.3)
    plt.show()


print(f"TensorFlow {tf.__version__}")
print("Setup complete")

## 1. TensorFlow in Colab

**TensorFlow** is Google's open-source library for computing with tensors. A **tensor** is a multi-dimensional array, like a numpy array, where every element has the same type. TensorFlow gives you **operations** that take tensors in and give tensors back (add, multiply, matrix multiply, sum and hundreds more), and it can run them on a CPU or a GPU.

**Keras** sits on top. It bundles operations into layers and models, and it is what most people write neural networks in. We reach it as `tf.keras`. Friday builds the first Keras model; today is the layer underneath.

Colab ships TensorFlow, so there is nothing to install. It runs its own Python, not the 3.15 we use for the ticket desk, because TensorFlow has no release for Python 3.15. The first thing a shared notebook prints is its versions, so anyone comparing results knows what ran.

In [ ]:
import sys
import tensorflow as tf          # the line every TensorFlow program starts with

print(f"Python      {sys.version.split()[0]}")
print(f"TensorFlow  {tf.__version__}")
print(f"Keras       {tf.keras.__version__}")
print(tf.config.list_physical_devices())     # the hardware TensorFlow can use

- `tf.__version__` is TensorFlow's version and `tf.keras.__version__` is Keras's. The numbers differ because Keras is its own project: since TensorFlow 2.16, `tf.keras` is **Keras 3**.
- `list_physical_devices()` lists the hardware TensorFlow can run on. We did not ask Colab for a GPU, so the list holds one CPU. A CPU is enough for everything today.

Try changing it: `tf.config.list_physical_devices("GPU")` prints `[]` on this runtime.

## 2. Tensors

Three things describe any tensor:

- **Rank:** how many axes it has. A single number is rank 0, a list is rank 1, a table is rank 2.
- **Shape:** the length of each axis, for example `(2, 2)`.
- **dtype** (data type): the one type every element shares, for example `float32` (32-bit decimal numbers) or `int32` (32-bit whole numbers).

`tf.constant` makes a tensor from Python values. Friday sets it against `tf.Variable`, the kind of tensor whose values can change.

In [ ]:
x = tf.constant([[1., 2.], [3., 4.]])
print(x)
print(type(x.numpy()).__name__)        # .numpy() copies the values into a numpy array

In [ ]:
scalar = tf.constant(7)
vector = tf.constant([1.0, 2.0, 3.0])
matrix = tf.constant([[1, 2], [3, 4]])

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(f"{name:<7} rank {t.ndim}  shape {str(t.shape):<7} dtype {t.dtype.name}")

- `print(x)` shows the values together with the shape and the dtype. TensorFlow always tracks both.
- Python ints became `int32` and Python floats became `float32`.
- Shape `()` means no axes at all: a single number.

A tensor is stricter than a numpy array in two ways:

In [ ]:
try:
    x[0, 0] = 5.0                          # fine on a numpy array
except TypeError as err:
    print(f"Edit in place:   {type(err).__name__}")

ints = tf.constant([1, 2])
floats = tf.constant([0.5, 0.5])
try:
    ints + floats                          # numpy would quietly turn the ints into floats
except Exception as err:
    print(f"int32 + float32: {type(err).__name__}")

print(f"After tf.cast:   {(tf.cast(ints, tf.float32) + floats).numpy()}")

- A tensor made with `tf.constant` is **immutable**: you cannot change one value inside it. To change values you build a new tensor.
- Mixing dtypes raises an error instead of converting silently. `tf.cast` converts on purpose. Section 4 needs it.

**Arithmetic.** `+`, `-`, `*` and `/` work **elementwise**: element by element, on tensors of the same shape. **Broadcasting** lets different shapes combine: an axis of length 1, or a missing axis, is stretched to match the other tensor, with the shapes lined up from the right. **Matrix multiplication** (rows times columns) is its own operation, `tf.matmul`.

In [ ]:
x = tf.constant([[1.0, 2.0], [3.0, 4.0]])
row = tf.constant([10.0, 20.0])
column = tf.constant([[1.0], [2.0], [3.0]])

print(f"x + row:\n{(x + row).numpy()}")              # the row is added to each row of x
print(f"x * 2:\n{(x * 2).numpy()}")
print(f"tf.matmul(x, x):\n{tf.matmul(x, x).numpy()}")
print(f"shape (3,) + shape (3, 1) -> {(tf.constant([1.0, 2.0, 3.0]) + column).shape}")

- `x + row`: the row, shape `(2,)`, is stretched to `(2, 2)` and added to each row of `x`.
- `x * 2`: a single number broadcasts to every element.
- `tf.matmul(x, x)`: rows times columns. The top-left value is `1*1 + 2*3 = 7`. Wednesday's layer (inputs times weights, plus a bias) is `tf.matmul(inputs, weights) + bias` in TensorFlow.
- A vector of shape `(3,)` plus a column of shape `(3, 1)` gives a full `(3, 3)` grid, with no error. Broadcasting is handy, and it is also an easy way to get a shape you did not mean.

Try changing it: make the row `[10.0, 20.0, 30.0]`. Lined up from the right, 2 and 3 do not match and neither is 1, so you get an error. Put it back.

## 3. Implementing basic graphs

TensorFlow 2 can run the same code two ways:

- **Eager execution**, the default: each operation runs the moment Python reaches it and returns a real value.
- **Graph execution**: TensorFlow first records the operations into a **graph** (the operations are the nodes, the tensors flowing between them are the edges), then runs the whole graph. A graph can be optimised, and it can run without Python, for example when a trained model is deployed to a server or a phone.

### 3.1 A function of tensors, run eagerly

An ordinary Python function. `x * w + b` is the weighted sum of Wednesday's neuron, without the activation. Multiply-then-add is called an **affine** function.

In [ ]:
def affine(x, w, b):
    return x * w + b          # one multiply, then one add

x = tf.constant([1.0, 2.0, 3.0])
w = tf.constant(2.0)
b = tf.constant(1.0)
print(affine(x, w, b).numpy())

`[3. 5. 7.]`: each element times 2, plus 1. Every line ran as soon as Python reached it, so a `print` anywhere inside the function would show a real value. That is why eager TensorFlow feels like numpy, and why it is easy to debug.

Try changing it: `w = tf.constant(-1.0)` gives `[ 0. -1. -2.]`. Put it back.

### 3.2 The same function with @tf.function

`tf.function` turns a Python function into a graph. The first time you call it, TensorFlow runs the Python body once and records every TensorFlow operation it meets into a graph. This is called **tracing**. Later calls with the same kind of input skip the Python and run the graph.

You can write `@tf.function` on the line above a `def`, or wrap a function that already exists, as here. Both do the same thing.

In [ ]:
graph_affine = tf.function(affine)      # same as writing @tf.function above def affine
print(graph_affine(x, w, b).numpy())

Same answer. What changed is underneath. `get_concrete_function` returns the graph traced for these inputs (a **concrete function**: one graph for one kind of input), and `.graph.get_operations()` lists its nodes.

In [ ]:
concrete = graph_affine.get_concrete_function(x, w, b)
for op in concrete.graph.get_operations():
    print(f"{op.name:<10} {op.type}")

Six nodes, name on the left and operation type on the right:

- One `Placeholder` per input, `x`, `w` and `b`. Inside a graph, a `Placeholder` is the node a function argument comes in through. It is not TensorFlow 1's `tf.placeholder` (section 5).
- `Mul` is the multiply and `AddV2` is the add.
- `Identity` hands the result out of the graph as the return value.

On a function this small the graph is no faster. Graphs pay off for code that runs many times and for saving a model to run without Python.

Try changing it: in 3.1, change `x * w` to `x * 2.0`, then rerun the 3.1 cell and both cells of 3.2. A `Const` node appears: a plain Python number is baked into the graph as a constant. Put it back and rerun the same cells.

### 3.3 Tracing

Python code inside a `tf.function` runs only while the function is traced. TensorFlow operations are recorded into the graph and run on every call. `print` shows the difference:

- a Python `print` runs during tracing only;
- `tf.print` is a TensorFlow operation, so it runs every time the graph runs.

In [ ]:
@tf.function
def double(x):
    print("Python print: tracing with shape", x.shape)
    tf.print("tf.print: running with", x)
    return x * 2

for t in [tf.constant([1.0, 2.0]), tf.constant([3.0, 4.0])]:    # same shape, same dtype
    double(t)

Two calls. The Python `print` ran once, on the first call, while TensorFlow traced. `tf.print` ran on both calls. If you put a Python `print` inside a `tf.function` to debug it and it prints only once, this is why: use `tf.print`.

Now a third call, with a shape this function has not seen:

In [ ]:
result = double(tf.constant([1.0, 2.0, 3.0]))     # shape (3,)
print(result.numpy())

A new shape means a new graph, so TensorFlow traced again and the Python `print` ran again. A new dtype does the same. Each trace costs time, so a function called with many different shapes keeps retracing. Section 4 fixes the input type up front.

Try changing it: make the input `tf.constant([1, 2])`. It is integers this time, so the Python `print` fires again: a new dtype traces a new graph, just like a new shape.

## 4. Reducing tensors of arbitrary shape

A **reduction** collapses one or more axes of a tensor into fewer values: a sum, a mean (average) or a max. The `axis` argument picks which axes disappear:

- `axis=None`, the default: every element, down to a single number (rank 0).
- `axis=0`: the first axis. `axis=-1`: the last axis (negative numbers count from the end).
- a tuple such as `axis=(1, 2)`: several axes at once.
- `keepdims=True`: each reduced axis stays, with length 1, so the result still lines up with the original.

We use small tensors we can check by eye, and one real batch: **MNIST**, 70,000 pictures of handwritten digits (60,000 for training, 10,000 for testing), each 28 by 28 pixels. Colab downloads it in a few seconds.

The pixels come back as `uint8`: whole numbers from 0 (background) to 255 (full ink). So the first cell **casts** them to `float32` and divides by 255, for two reasons. The mean of whole numbers comes back as a whole number, with the fraction cut off. And the reducer we write at the end of this section accepts `float32` only.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(f"x_train: shape {x_train.shape}, dtype {x_train.dtype}")

counts = tf.constant([1, 2])
print(f"Mean of [1, 2] as int32:   {tf.reduce_mean(counts).numpy()}")      # the .5 is cut off
print(f"Mean of [1, 2] as float32: {tf.reduce_mean(tf.cast(counts, tf.float32)).numpy()}")

images = tf.cast(x_train, tf.float32) / 255.0      # cast first, then scale 0..255 down to 0..1
print(f"images:  shape {images.shape}, dtype {images.dtype.name}")

- `x_train` is a numpy array of 60,000 images. TensorFlow operations accept numpy arrays and return tensors.
- The integer mean of `[1, 2]` is `1`, not `1.5`. On the pixels the same thing would quietly cut off the fraction of every average.
- `images` has the same shape, now `float32` with values from 0 to 1.

**Every rank.** A reduction works on a tensor of any rank. With no `axis`, each one collapses everything to a single number:

In [ ]:
small = [tf.constant(4.0),                                   # rank 0
         tf.constant([1.0, 2.0, 6.0]),                       # rank 1
         tf.constant([[1.0, 2.0], [3.0, 10.0]]),             # rank 2
         tf.constant([[[1., 2., 3.], [4., 5., 6.]],
                      [[7., 8., 9.], [10., 11., 12.]]])]     # rank 3

for t in small:
    print(f"rank {t.ndim}  shape {str(t.shape):<10} sum {tf.reduce_sum(t).numpy():5.1f}"
          f"  mean {tf.reduce_mean(t).numpy():4.1f}  max {tf.reduce_max(t).numpy():5.1f}")

t = small[3]          # keep the rank-3 tensor, shape (2, 2, 3), for the next cells

**Choosing axes.** The same sum on the rank-3 tensor of shape `(2, 2, 3)`: think of it as 2 blocks, each with 2 rows of 3 numbers.

In [ ]:
for axis in [None, 0, -1, (1, 2)]:
    total = tf.reduce_sum(t, axis=axis)
    print(f"axis={axis}: shape {total.shape}")
    print(total.numpy())
    print()

- `axis=None`: everything, `78.0`, shape `()`.
- `axis=0` adds the two blocks together, number by number: shape `(2, 3)`.
- `axis=-1` sums each row of 3: shape `(2, 2)`.
- `axis=(1, 2)` sums everything inside each block: one number per block, shape `(2,)`.

The rule: **the axes you name disappear from the shape.** `tf.reduce_mean` and `tf.reduce_max` follow the same rule.

**keepdims.** Sometimes you want the reduced result to line up with the original again, for example to subtract each row's mean from that row:

In [ ]:
row_means = tf.reduce_mean(t, axis=-1, keepdims=True)
print(f"keepdims=True:  shape {row_means.shape}")
print(f"keepdims=False: shape {tf.reduce_mean(t, axis=-1).shape}")
print((t - row_means).numpy()[0])        # first block: every row minus its own mean

With `keepdims=True` the reduced axis stays as length 1, so `(2, 2, 1)` broadcasts back against `(2, 2, 3)` and each row loses its own mean. Subtracting the mean like this is a common step before training (it is called **centring** the data).

Try changing it: drop `keepdims=True` from the first line. Shapes `(2, 2, 3)` and `(2, 2)` do not line up from the right, so the subtraction raises an error. Put it back.

**`tf.argmax` is not a fourth reducer.** It returns the *position* of the largest value, not the value, and it is stricter:

- it takes **one integer axis**, never a tuple;
- leave `axis` out and it uses axis 0: it does **not** search the whole tensor;
- it has no `keepdims`.

In [ ]:
m = tf.constant([[3., 9., 2.],
                 [8., 1., 7.]])
print(f"tf.argmax(m):             {tf.argmax(m).numpy()}")              # axis 0: winning row, per column
print(f"tf.argmax(m, axis=1):     {tf.argmax(m, axis=1).numpy()}")      # winning column, per row
print(f"tf.reduce_max(m, axis=1): {tf.reduce_max(m, axis=1).numpy()}")  # the winning values

- Down each column: 8 beats 3 (row 1), 9 beats 1 (row 0), 7 beats 2 (row 1), so `tf.argmax(m)` is `[1 0 1]`.
- Across each row the largest value sits in column 1 (the 9) and column 0 (the 8): `[1 0]`.
- `reduce_max` gives the values themselves, `[9. 8.]`.

In a classifier, `argmax` over the scores for each class is how the predicted class is picked.

**The MNIST batch.** `images` has shape `(60000, 28, 28)`: axis 0 says which image, axes 1 and 2 are a pixel's row and column.

In [ ]:
print(f"Mean of every pixel in every image: {tf.reduce_mean(images).numpy():.4f}")

mean_image = tf.reduce_mean(images, axis=0)                      # average over the 60000 images
brightest = tf.reduce_max(images, axis=(1, 2))                   # one value per image
position = tf.argmax(tf.reshape(images, (-1, 784)), axis=1)      # 28 x 28 = 784 pixels in one row

print(f"mean_image {mean_image.shape}   brightest {brightest.shape}   position {position.shape}")
print(f"Brightest pixel of the first 5 images: {brightest[:5].numpy()}")
print(f"Its position in the row of 784:        {position[:5].numpy()}")

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(mean_image.numpy(), cmap="gray")
plt.title("The mean of 60,000 digits")
plt.axis("off")
plt.show()

- The mean of every pixel is about `0.13`, close to 0: most of each picture is empty background.
- `axis=0` gives `(28, 28)`: one average image. It is bright in the middle, where digits are drawn, and black at the edges.
- `axis=(1, 2)` gives `(60000,)`: one number per image. The first five each reach `1.0`, a pixel at full ink.
- `tf.argmax` takes one axis only, so each image is first reshaped into one row of 784 pixels (`-1` lets TensorFlow work out the 60000), then searched along `axis=1`. Position `161` is row 5, column 21 of the first image, because `161 = 5 * 28 + 21`.

Try changing it: in the `mean_image` line, use `axis=(1, 2)` and rerun: the shape is `(60000,)`, the average brightness of each image. Then put back `axis=0`: `(28, 28)`, one average image. Add `keepdims=True` to either and you get `(60000, 1, 1)` and `(1, 28, 28)`.

**One function for any shape.** In 3.3 every new shape traced a new graph. `input_signature` fixes the input type up front: `tf.TensorSpec(shape=None, dtype=tf.float32)` means "a `float32` tensor of any shape", so one graph serves every rank.

In [ ]:
@tf.function(input_signature=[tf.TensorSpec(shape=None, dtype=tf.float32)])
def summarize(x):
    print("tracing")                    # Python print: runs only while tracing
    return tf.reduce_sum(x), tf.reduce_mean(x), tf.reduce_max(x)

for tensor in small[:3] + [images]:
    total, mean, largest = summarize(tensor)
    print(f"shape {str(tensor.shape):<15} sum {total.numpy():<11.6g} "
          f"mean {mean.numpy():<8.4g} max {largest.numpy():.4g}")

`tracing` printed once: one graph reduced a single number, a vector, a matrix and the whole batch of 60,000 images. Now pass it the pixels without the cast:

In [ ]:
raw = tf.constant(x_train)            # the pixels as a uint8 tensor
try:
    summarize(raw)
except TypeError as err:
    print(f"{type(err).__name__}: {str(err)[:120]}...")

The signature asks for `float32` and this tensor is `uint8`, so the function refuses it instead of guessing. The fix is the cast from the top of this section.

Try changing it: call `summarize(x_train)` with the numpy array instead. A numpy array is converted to `float32` for you, but nothing divides by 255, so the mean comes back as about `33.3` instead of `0.13`.

## 5. Graphs, sessions, TensorBoard

"Graphs" and "sessions" are words from **TensorFlow 1** (version 1.0 came out in February 2017). Old tutorials and forum answers are full of them, so you need to recognise them, but you never write them in TensorFlow 2.

### 5.1 TensorFlow 1 sessions, as history

In TensorFlow 1 you built the whole graph first, with a **placeholder** standing in for each input. Then a **session** ran the graph, and you passed the real values in a `feed_dict` (a dictionary from each placeholder to its value).

**TensorFlow 1 (history, do not run)**

```python
# TensorFlow 1 (history, do not run)
x = tf.placeholder(tf.float32, shape=[None])
y = x * 2.0 + 1.0
with tf.Session() as sess:
    print(sess.run(y, feed_dict={x: [1.0, 2.0, 3.0]}))
```

TensorFlow 2.0 (September 2019) made eager execution the default and dropped both names. Pasted into a code cell, this block stops on its first line with an `AttributeError`, because TensorFlow 2 has no `tf.placeholder`. The old API still lives in `tf.compat.v1`, but only as a bridge for teams moving old code forward; new code does not use it.

The TensorFlow 2 version: a `tf.function` replaces the session, and an ordinary argument replaces the placeholder.

In [ ]:
@tf.function
def affine_tf2(x):
    return x * 2.0 + 1.0

print(affine_tf2(tf.constant([1.0, 2.0, 3.0])).numpy())

`[3. 5. 7.]`, the same answer the session would print, with no graph-building step, no session and no `feed_dict`.

| TensorFlow 1 | TensorFlow 2 |
|---|---|
| build a graph, then `session.run(...)` | call a `tf.function` (or just run eagerly) |
| `tf.placeholder` | a function argument |
| `feed_dict={x: values}` | `f(values)` |

Try changing it: make the input `[[1.0, 2.0], [3.0, 4.0]]`. You get `[[3. 5.] [7. 9.]]`: an argument takes any shape, and the function traces again for the new one.

### 5.2 A graph in TensorBoard

**TensorBoard** is TensorFlow's dashboard: a web page that reads log files and draws them. Today we use its **Graphs** tab to look at the graph from section 3. Week 7 uses it to watch a model train.

Getting a graph into it takes four steps, in this order:

1. `tf.summary.create_file_writer("logs/graph")` makes a **writer**, which writes **event files** (TensorBoard's log format) into that folder.
2. `tf.summary.trace_on(graph=True)` starts recording graphs.
3. Call the function once. It is traced now, and the trace is recorded.
4. `tf.summary.trace_export(...)`, inside `with writer.as_default():`, writes the recorded graph through the writer and stops recording.

The graph is recorded when a function is traced, and `graph_affine` was already traced in 3.2. So we wrap 3.1's `affine` in a fresh `tf.function`.

In [ ]:
basic_graph = tf.function(affine)          # a fresh wrap of 3.1's affine, not traced yet

writer = tf.summary.create_file_writer("logs/graph")
tf.summary.trace_on(graph=True)
basic_graph(tf.constant([1.0, 2.0, 3.0]), tf.constant(2.0), tf.constant(1.0))
with writer.as_default():
    tf.summary.trace_export(name="basic_graph", step=0)

The cell prints nothing. It wrote an event file under `logs/graph`, which you can see in Colab's Files pane on the left.

Now load TensorBoard and point it at the `logs` folder. Lines that start with `%` are **magics**: commands for the notebook itself, not Python. `%load_ext tensorboard` loads TensorBoard, and `%tensorboard --logdir logs` opens it inside the cell's output.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

- The Graphs tab opens with two boxes, `graph_1` and `graph_2`. `graph_1` is our function: double-click it to open it. Inside are the three inputs `x`, `w` and `b` feeding `mul`, then `add`, then `Identity`: the same nodes `get_operations()` listed in 3.2. The small `graph_2` was recorded by the export call itself; ignore it.
- The Graphs tab shows structure: which operations, joined how. It says nothing about whether a model is learning. That view is week 7's.
- An empty Graphs tab usually means `trace_export` ran outside the `with writer.as_default():` block, or `%tensorboard` ran before `%load_ext tensorboard`.

Try changing it: use `--logdir logs/nothing`. TensorBoard opens with no Graphs tab, because that folder holds no event file. Put it back.

## Recap

| Idea | Type | Good at | Watch out for |
|---|---|---|---|
| Tensor (`tf.constant`) | data | one shape and one dtype, runs on CPU or GPU | immutable; mixing dtypes is an error, so `tf.cast` |
| Eager execution | way of running | every line returns a value you can print | the default; easy to debug, harder to export |
| `tf.function` | way of running (graph) | trace once, rerun the graph, run without Python | Python code runs only while tracing; a new shape or dtype retraces |
| `reduce_sum`, `reduce_mean`, `reduce_max` | operation | collapse any axes of any shape | named axes disappear unless `keepdims=True`; cast `uint8` first |
| `tf.argmax` | operation | the position of the largest value | one integer axis, default 0, no `keepdims` |
| Sessions and placeholders | TensorFlow 1 history | reading old tutorials | never run them; `tf.compat.v1` is for migrating old code |
| TensorBoard Graphs tab | tool | seeing the operations of a traced function | `trace_export` inside `with writer.as_default():`; `%load_ext` before `%tensorboard` |

**Quick check**
1. Name two ways a TensorFlow tensor made with `tf.constant` differs from a numpy array.
2. What does `@tf.function` do the first time you call a function, and what happens on later calls?
3. A `tf.function` holds a Python `print` and a `tf.print`. You call it three times: shape `(2,)`, shape `(2,)`, then shape `(3,)`. How many times does each one print?
4. `images` has shape `(60000, 28, 28)`. What shapes do `tf.reduce_mean(images, axis=0)` and `tf.reduce_max(images, axis=(1, 2))` return, and why did we cast the pixels to `float32` first?
5. An old tutorial uses `tf.Session`, `tf.placeholder` and `feed_dict`. What replaces each in TensorFlow 2?
6. List the steps that put a function's graph into TensorBoard's Graphs tab.